# Preparación final de los CSV
Grupo 1, Machine Learning, UPCH 2026-II.

César Alejandro Aarón Apcho Meneses, Paola Andrea Centeno Bazán, Anderson Josué Delerna Infantes y Jimena Natalia Vega Sánchez.

La página Evaluation mostraba `ID,Class` y un ejemplo desde 1. El evaluador rechazó ese formato y aceptó **`id,Class`**, con ID de **0 a 32767**. Esta celda convierte los cinco candidatos del notebook 02 conservando el orden de `test_x.h5` y las clases predichas. El modelo no vuelve a entrenarse.

Descargar el repositorio completo y ejecutar desde `notebooks/`. Solo requiere Python estándar. Los CSV finales están en `results/cnn/`. [Registro de los cinco envíos](../docs/ENTREGA_KAGGLE.md).

Las salidas se restauran de la ejecución local verificada en esta conversación, realizada antes de que el entorno dejara de estar disponible. Sus cinco SHA-256 se comprobaron nuevamente al reconstruir los archivos desde GitHub.


In [1]:
from pathlib import Path
import csv
import hashlib
import json
import shutil

# Localizar la raíz del repositorio en Jupyter, Colab o Kaggle.
candidatas = [Path.cwd(), Path.cwd().parent]
if Path('/kaggle/working').exists():
    candidatas += [p.parent.parent.parent for p in Path('/kaggle/input').rglob('candidato_01_base.csv')]
raices = [p for p in candidatas if (p / 'results/cnn/candidato_01_base.csv').is_file()]
if not raices:
    raise FileNotFoundError('Clonar el repositorio y ejecutar este cuaderno desde su carpeta notebooks.')
raiz = raices[0]
origen = raiz / 'results/cnn'
salida = Path('/kaggle/working/results/cnn') if Path('/kaggle/working').exists() else origen
salida.mkdir(parents=True, exist_ok=True)
registros = []
for experimento in ['01_base', '02_dropout', '03_aumento', '04_profunda', '05_batchnorm']:
    fuente = origen / f'candidato_{experimento}.csv'
    with fuente.open(newline='', encoding='utf-8') as archivo:
        lector = csv.DictReader(archivo)
        assert lector.fieldnames == ['ID', 'Class']
        filas = list(lector)
    assert len(filas) == 32768
    assert [int(f['ID']) for f in filas] == list(range(1, 32769))
    assert {f['Class'] for f in filas} <= {'0', '1'}
    destino = salida / f'submission_{experimento}.csv'
    with destino.open('w', newline='', encoding='utf-8') as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(['id', 'Class'])
        escritor.writerows((i, int(f['Class'])) for i, f in enumerate(filas))
    with destino.open(newline='', encoding='utf-8') as archivo:
        lector = csv.DictReader(archivo)
        assert lector.fieldnames == ['id', 'Class']
        comprobacion = list(lector)
    assert [int(f['id']) for f in comprobacion] == list(range(32768))
    assert [int(f['Class']) for f in comprobacion] == [int(f['Class']) for f in filas]
    firma = hashlib.sha256(destino.read_bytes()).hexdigest()
    registros.append(dict(experimento=experimento, archivo=destino.name, filas=32768, sha256=firma))
    print(f'{destino.name}: 32768 filas, id=0..32767, clases verificadas. SHA-256 {firma}')
shutil.copyfile(salida / 'submission_02_dropout.csv', salida / 'submission.csv')
(salida / 'archivos_enviados.json').write_text(json.dumps(registros, indent=2, ensure_ascii=False), encoding='utf-8')
print('submission.csv coincide con 02_dropout, seleccionado por validación antes de los envíos.')
print('La generación de estos archivos no realiza nuevos envíos. Las puntuaciones reales están en docs/ENTREGA_KAGGLE.md.')


submission_01_base.csv: 32768 filas, id=0..32767, clases verificadas. SHA-256 d6d2301d3ecc5fd7fb6353df79585af62cbabf464ab87db3659064670dc6dbb7
submission_02_dropout.csv: 32768 filas, id=0..32767, clases verificadas. SHA-256 09d21c7ca0abd86b25bb121966dd6a55f8fb3ebb51be875d5fa511dbd4543af0
submission_03_aumento.csv: 32768 filas, id=0..32767, clases verificadas. SHA-256 8cdf60923ecee7e2a55a14e49c252c7f7c76aa2b798e458783d06e040caf5899
submission_04_profunda.csv: 32768 filas, id=0..32767, clases verificadas. SHA-256 22c08914bf40f55e7f463ccb7b52f7556dade97146cb1d01719748a30662a9fe
submission_05_batchnorm.csv: 32768 filas, id=0..32767, clases verificadas. SHA-256 bebf2a2fb8b98f54f444774ed84612e17b040fca9bc2a93d6610c95f4c47ac2f
submission.csv coincide con 02_dropout, seleccionado por validación antes de los envíos.
La generación de estos archivos no realiza nuevos envíos. Las puntuaciones reales están en docs/ENTREGA_KAGGLE.md.
